# Extra results: weight compression, latent interpolation

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys


sys.path.append("..")

os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [ ]:
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from copy import deepcopy

import matplotlib.pyplot as plt

from neugk.pinc.neural_fields.nf_utils import compress_weights, load_nf, sample_field
from neugk.pinc.neural_fields.gk_losses import integral_losses
from neugk.pinc.neural_fields.data import CycloneNFDataset, CycloneNFDataLoader
from neugk.pinc.neural_fields.nf_train import eval_diagnose, train_density

In [ ]:
def compute_metrics(gt, pred):
    gt, pred = gt.cpu(), pred.cpu()
    l1 = torch.mean(torch.abs(gt - pred)).item()
    mse = ((pred.cpu() - gt.cpu()) ** 2).mean()
    psnr = 10 * torch.log10(gt.max() ** 2 / mse**2)
    return l1, psnr

In [ ]:
device = "cuda"

TIMESTEPS = [100, 120, 140, 160, 180, 200]
TRAJECTORIES = [
    "iteration_13",
    "iteration_115",
    "iteration_131",
    "iteration_134",
    "iteration_146",
    "iteration_148",
    "iteration_160",
    "iteration_200",
    "iteration_210",
    "iteration_212",
]

DELTA = TIMESTEPS[1] - TIMESTEPS[0]

DATASETS = {}
for traj in TRAJECTORIES:
    DATASETS[traj] = {}
    for t in TIMESTEPS:
        time_a = t
        time_b = time_a + DELTA
        time_c = time_a + DELTA // 2
        DATASETS[traj][t] = {
            "a": CycloneNFDataset(
                trajectory=f"{traj}.h5",
                timesteps=time_a,
                normalize="zscore",
                normalize_coords=False,
                realpotens=True,
            ),
            "b": CycloneNFDataset(
                trajectory=f"{traj}.h5",
                timesteps=time_b,
                normalize="zscore",
                normalize_coords=False,
                realpotens=True,
            ),
            "c": CycloneNFDataset(
                trajectory=f"{traj}.h5",
                timesteps=time_c,
                normalize="zscore",
                normalize_coords=False,
                realpotens=True,
            ),
        }

In [ ]:
MODEL = "mlp"
POSTFIX = "x1163"

missing = {}
missing_int = {}
for traj in TRAJECTORIES:
    missing[traj] = []
    missing_int[traj] = []
    for t in TIMESTEPS:
        ckp_name = f"{MODEL}_{traj}_t{t}_{POSTFIX}.pt"
        if not os.path.exists(f"../nf_ckps/{ckp_name}"):
            missing[traj].append(t)
        if not os.path.exists(f"../nf_ckps/int_{ckp_name}"):
            missing_int[traj].append(t)
missing_int

In [ ]:
MODELS = {}
for traj in TRAJECTORIES:
    MODELS[traj] = {}
    for t in TIMESTEPS:
        ckp_name = f"int_{MODEL}_{traj}_t{t}_{POSTFIX}.pt"
        MODELS[traj][t] = load_nf(f"../nf_ckps/{ckp_name}", device).to(device).eval()

In [ ]:
model = MODELS["iteration_13"][200]

n_params = sum(p.numel() for p in model.parameters())
model_size = sum(p.nbytes for p in model.parameters())
compression = DATASETS["iteration_13"][200]["a"].full_df.nbytes / model_size
print(f"Params: {n_params / 1e3:.2f}k, compression: {compression:.2f}x")

## Weight space compression

In [ ]:
eval_diagnose(
    model=model,
    data=DATASETS["iteration_13"][200]["a"],
    T=None,
    device=device,
    metrics_only=True,
)

### ZFP

In [ ]:
model_zfp, original_size, compressed_size = compress_weights(
    model, method="zfp", tolerance=0.01
)

weight_compression = original_size / compressed_size
n_params = sum(p.numel() for p in model.parameters())
data_compression = DATASETS["iteration_13"][200]["a"].full_df.numel() / n_params
total_compression = data_compression * weight_compression
print(f"Params: {n_params / 1e3:.2f}k")
print(f"Model original: {original_size / (1024 ** 2):.2f}MB")
print(f"Model compressed: {compressed_size / (1024 ** 2):.2f}MB")
print(f"Weight compression: {weight_compression:.2f}x")
print(f"Data compression: {data_compression:.2f}x --> {total_compression:.2f}x")
print()
eval_diagnose(
    model=model_zfp,
    data=DATASETS["iteration_13"][200]["a"],
    T=None,
    device=device,
    metrics_only=True,
)

### ZipNN

In [ ]:
model_zipnn, original_size, compressed_size = compress_weights(model, method="zipnn")

weight_compression = original_size / compressed_size
n_params = sum(p.numel() for p in model.parameters())
data_compression = DATASETS["iteration_13"][200]["a"].full_df.numel() / n_params
total_compression = data_compression * weight_compression
print(f"Params: {n_params / 1e3:.2f}k")
print(f"Model original: {original_size / (1024 ** 2):.2f}MB")
print(f"Model compressed: {compressed_size / (1024 ** 2):.2f}MB")
print(f"Weight compression: {weight_compression:.2f}x")
print(f"Data compression: {data_compression:.2f}x --> {total_compression:.2f}x")
print()
eval_diagnose(
    model=model_zipnn,
    data=DATASETS["iteration_13"][200]["a"],
    T=None,
    device=device,
    metrics_only=True,
)

### Float8 quantize

In [ ]:
model_quant, original_size, compressed_size = compress_weights(
    model, method="quantize8"
)

weight_compression = original_size / compressed_size
n_params = sum(p.numel() for p in model.parameters())
data_compression = DATASETS["iteration_13"][200]["a"].full_df.numel() / n_params
total_compression = data_compression * weight_compression
print(f"Params: {n_params / 1e3:.2f}k")
print(f"Model original: {original_size / (1024 ** 2):.2f}MB")
print(f"Model compressed: {compressed_size / (1024 ** 2):.2f}MB")
print(f"Weight compression: {weight_compression:.2f}x")
print(f"Data compression: {data_compression:.2f}x --> {total_compression:.2f}x")
print()
eval_diagnose(
    model=model_quant,
    data=DATASETS["iteration_13"][200]["a"],
    T=None,
    device=device,
    metrics_only=True,
)

### Quantitative eval

In [ ]:
results = []
with torch.no_grad():
    for traj in TRAJECTORIES:
        for t in TIMESTEPS:
            data = DATASETS[traj][t]["a"].to(device)

            model_base = MODELS[traj][t].to(device)

            # baseline
            losses_base = integral_losses(model_base, data=data, device=device)
            mse_base = losses_base["df loss"]
            psnr_base = 10 * torch.log10(data.full_df.max() ** 2 / mse_base**2)
            l1q_base = losses_base["flux loss"]
            l1p_base = losses_base["phi loss"]

            # ZFP
            model_zfp, original_size, compressed_size = compress_weights(
                model_base, method="zfp", tolerance=0.001
            )
            cr_zfp = original_size / compressed_size
            losses_zfp = integral_losses(model_zfp, data=data, device=device)
            mse_zfp = losses_zfp["df loss"]
            psnr_zfp = 10 * torch.log10(data.full_df.max() ** 2 / mse_zfp**2)
            l1q_zfp = losses_zfp["flux loss"]
            l1p_zfp = losses_zfp["phi loss"]

            # ZipNN
            model_zipnn, original_size, compressed_size = compress_weights(
                model_base, method="zipnn"
            )
            cr_zip = original_size / compressed_size
            losses_zip = integral_losses(model_zipnn, data=data, device=device)
            mse_zip = losses_zip["df loss"]
            psnr_zip = 10 * torch.log10(data.full_df.max() ** 2 / mse_zip**2)
            l1q_zip = losses_zip["flux loss"]
            l1p_zip = losses_zip["phi loss"]

            results.append(
                {
                    "Trajectory": traj,
                    "Timestep": t,
                    "PSNR": psnr_base.item(),
                    "L1Q": l1q_base.item(),
                    "L1P": l1p_base.item(),
                    "ZFP CR": cr_zfp,
                    "ZFP PSNR": psnr_zfp.item(),
                    "ZFP L1Q": l1q_zfp.item(),
                    "ZFP L1P": l1p_zfp.item(),
                    "ZipNN CR": cr_zip,
                    "ZipNN PSNR": psnr_zip.item(),
                    "ZipNN L1Q": l1q_zip.item(),
                    "ZipNN L1P": l1p_zip.item(),
                }
            )

df_metrics = pd.DataFrame(results)

In [ ]:
df_metrics["ZFP L1P"].mean(), df_metrics["L1P"].mean()

In [ ]:
row = {
    "Extra CR": (
        f"{df_metrics['ZFP CR'].mean():.1f}$\\times$",
        f"{df_metrics['ZipNN CR'].mean():.1f}$\\times$",
    ),
    r"$\Delta$ PSNR ($\boldsymbol{f}$)": (
        f"{((df_metrics['PSNR'] - df_metrics['ZFP PSNR']) / df_metrics['ZFP PSNR'] * 100).mean():.5f}\\%",
        f"{((df_metrics['PSNR'] - df_metrics['ZipNN PSNR']) / df_metrics['ZipNN PSNR'] * 100).mean():.3f}\\%",
    ),
    r"$\Delta$ L1 ($Q$)": (
        f"{((df_metrics['L1Q'] - df_metrics['ZFP L1Q']) / df_metrics['ZFP L1Q'] * 100).mean():.3f}\\%",
        f"{((df_metrics['L1Q'] - df_metrics['ZipNN L1Q']) / df_metrics['ZipNN L1Q'] * 100).mean():.3f}\\%",
    ),
    r"$\Delta$ L1 ($\boldsymbol{{\phi}}$)": (
        f"{((df_metrics['L1P'] - df_metrics['ZFP L1P']) / df_metrics['ZFP L1P'] * 100).mean():.3f}\\%",
        f"{((df_metrics['L1P'] - df_metrics['ZipNN L1P']) / df_metrics['ZipNN L1P'] * 100).mean():.3f}\\%",
    ),
}

# Build table
table = [
    r"\begin{wraptable}{r}{0.4\linewidth}",
    r"\centering",
    r"\vspace{-16px}",
    r"\caption{Hybrid compression.\label{tab:hybrid}}",
    r"\vspace{-8px}",
    r"\begin{tabular}{lcccc}",
    r"\toprule",
    r"Metric & & \texttt{ZFP} & \texttt{ZipNN} \\",
    r"\midrule",
]

for metric, (zfp_val, zip_val) in row.items():
    arrow = r"$\uparrow$" if "CR" in metric or "PSNR" in metric else r"$\downarrow$"
    table.append(f"{metric} & {arrow} & {zfp_val} & {zip_val} \\\\")

table += [
    r"\bottomrule",
    r"\end{tabular}",
    r"\vspace{-8px}",
    r"\end{wraptable}",
]

table = "\n".join(table)
print(table)

## Interpolation experiments

In [ ]:
def interpolate_models(model_a, model_b, alpha: float = 0.5):
    if alpha == 0.0:
        return deepcopy(model_a)
    if alpha == 1.0:
        return deepcopy(model_b)
    model_interp = deepcopy(model_a)
    model_interp.load_state_dict(model_a.state_dict())
    state_dict_a = model_a.state_dict()
    state_dict_b = model_b.state_dict()
    state_dict_interp = {}
    for key in state_dict_a.keys():
        state_dict_interp[key] = (1 - alpha) * state_dict_a[key] + alpha * state_dict_b[
            key
        ]
    model_interp.load_state_dict(state_dict_interp)
    return model_interp

### "Metalearning" a shared initialization

In [ ]:
from neugk.pinc.neural_fields.models.mlp import MLPNF


total = len(TRAJECTORIES) * len(TIMESTEPS) * 4
with tqdm(total=total) as pbar:
    for traj in TRAJECTORIES:
        model0 = MLPNF(
            in_dim=5,
            out_dim=2,
            n_layers=5,
            dim=64,
            act_fn=torch.nn.SiLU,
            use_checkpoint=False,
            skips=True,
            embed_type="discrete",
        )
        optim = torch.optim.AdamW(model0.parameters(), 5e-3)
        for timestep in TIMESTEPS:
            for _ in range(4):
                data = CycloneNFDataset(
                    trajectory=traj,
                    timesteps=timestep,
                    normalize="zscore",
                    normalize_coords=False,
                    realpotens=True,
                )
                loader = CycloneNFDataLoader(data, 5096, preload=True, shuffle=True)
                model0, _, losses = train_density(
                    model0,
                    n_epochs=2,
                    data=data,
                    loader=loader,
                    device=device,
                    optim=optim,
                    use_tqdm=False,
                    use_print=False,
                )
                pbar.set_postfix(
                    {
                        "traj": traj,
                        "MSE": losses["train/loss"],
                        "PSNR": losses["val/df psnr"].item(),
                    }
                )
                pbar.update(1)
        torch.save(
            model0.state_dict(), f"../nf_shared_init/{traj.replace('.h5', '')}.pth"
        )

In [ ]:
TIME_A = TIMESTEPS[0]
TIME_B = TIME_A + DELTA
TIME_C = TIME_A + DELTA // 2

TRAJ = "iteration_13"

ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0]


GT_DFS = [
    CycloneNFDataset(
        f"{TRAJ}.h5", timesteps=int(TIME_A + DELTA * a), realpotens=True
    ).full_df
    for a in ALPHAS
]
interp_models = {"a": MODELS[TRAJ][TIME_A], "b": MODELS[TRAJ][TIME_B]}

In [ ]:
def plot_fields_grid(
    dfs,
    suptitle=None,
    projections=None,
    titles=None,
    row_labels=None,
    cmap="RdBu_r",
    use_labels=False,
):
    phy_times = np.loadtxt(
        "/restricteddata/ukaea/gyrokinetics/raw/iteration_13/time.dat"
    )
    all_labels = [r"$v_{\parallel}$", r"$\mu$", r"$s$", r"$x$", r"$y$"]

    if isinstance(dfs, list):
        dfs = torch.stack(dfs, 0)

    num_cols = dfs.shape[0]
    if projections is None:
        projections = [(0, 1, 2), (1, 3, 4)]
    num_rows = len(projections)

    fig, ax = plt.subplots(num_rows, num_cols, figsize=(2 * num_cols, 0.94 * num_rows))
    if num_rows == 1:
        ax = ax[None, :]
    if num_cols == 1:
        ax = ax[:, None]

    for i, proj in enumerate(projections):
        data_list = [dfs[j].mean(0).mean(proj) for j in range(num_cols)]
        vmin, vmax = min(d.min() for d in data_list), max(d.max() for d in data_list)
        for j in range(num_cols):
            ax[i, j].matshow(
                data_list[j], cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto"
            )
            ax[i, j].set_xticks([])
            ax[i, j].set_yticks([])
            im = ax[i, j].get_images()[0]
            # extent = im.get_extent()
            # ax[i, j].set_aspect(abs((extent[1]-extent[0])/(extent[3]-extent[2]) / 2.0))
            # ax[i, j].axis("off")
            # Column titles (first row)
            if i == 0 and titles is not None:
                ax[i, j].set_title(
                    rf"$t={phy_times[titles[j]]:.1f}R/V_{{\mathrm{{r}}}}$", fontsize=18
                )

        # Row labels on the left (once per row)
        if row_labels is not None:
            ax[i, 0].set_ylabel(row_labels[i], fontsize=16)

        # Axis labels corresponding to the projection
        if use_labels:
            label_text = "/".join([all_labels[o] for o in range(5) if o not in proj])
            fig.text(
                0.02,
                1.0 * (num_rows - i) / num_rows - 0.25,
                label_text,
                va="center",
                ha="center",
                fontsize=18,
            )

    fig.subplots_adjust(
        left=0.06, right=0.96, top=0.95, bottom=0, wspace=0.0, hspace=0.0
    )

    if suptitle is not None:
        fig.text(0.97, 0.5, suptitle, va="center", ha="left", fontsize=20, rotation=-90)
    fig.patch.set_alpha(0.0)
    return fig

In [ ]:
dfs = []
with torch.no_grad():
    for a in ALPHAS:
        model_a = interpolate_models(interp_models["a"], interp_models["b"], alpha=a)
        df = sample_field(
            model_a, DATASETS[TRAJ][TIME_A]["a"], device, timestep=None
        ).cpu()
        dfs.append(df)
dfs = torch.stack(dfs, 0)
fig = plot_fields_grid(
    dfs,
    titles=[int(TIME_A + DELTA * a) for a in ALPHAS],
    suptitle="Weight space",
    use_labels=True,
)
fig.savefig("figures/weight_interp.pdf", bbox_inches="tight", transparent=True)

In [ ]:
dfs = []
with torch.no_grad():
    for a in ALPHAS:
        df = (1 - a) * DATASETS[TRAJ][TIME_A]["a"].full_df + a * DATASETS[TRAJ][TIME_A][
            "b"
        ].full_df
        dfs.append(df)
dfs = torch.stack(dfs, 0)
fig = plot_fields_grid(dfs, suptitle="Data space", use_labels=True)
fig.savefig("figures/data_interp.pdf", bbox_inches="tight", transparent=True)

In [ ]:
dfs = []
with torch.no_grad():
    for i in range(len(ALPHAS)):
        df = GT_DFS[i]
        dfs.append(df)
dfs = torch.stack(dfs, 0)
fig = plot_fields_grid(dfs, suptitle="Ground truth", use_labels=True)
fig.savefig("figures/gt_interp.pdf", bbox_inches="tight", transparent=True)

In [ ]:
DELTA = TIMESTEPS[1] - TIMESTEPS[0]

### Autoencoders

In [ ]:
from neugk.pinc.autoencoders.ae_utils import load_autoencoder

ae_root = "/system/user/publicwork/gyrokinetics_checkpoints/diff_ae"
file = "ggall/ae_pre/ckp.pth"

ae_ckp = f"{ae_root}/{file}"
ae, _, cfg = load_autoencoder(ae_ckp, device=device)
ae = ae.to(device).eval()

file = "ggall/vqvae_base/best.pth"
vqvae_ckp = f"{ae_root}/{file}"
vqvae, _, cfg = load_autoencoder(vqvae_ckp, device=device)
vqvae = vqvae.to(device).eval()

### Interpolation table

In [ ]:
import pandas as pd

from neugk.dataset import CycloneAEDataset


def compute_metrics(gt, pred):
    gt, pred = gt.cpu(), pred.cpu()
    l1 = torch.mean(torch.abs(gt - pred)).item()
    mse = ((pred.cpu() - gt.cpu()) ** 2).mean()
    psnr = 10 * torch.log10(gt.max() ** 2 / mse**2)
    return l1, psnr


def autoencoder_interp_latent(
    model, lat_a, lat_b, condition_a, condition_b, pad_axes, alpha
):
    zdf = (1 - alpha) * lat_a + alpha * lat_b
    condition = (1 - alpha) * condition_a + alpha * condition_b
    if hasattr(model, "quantize"):
        zdf = model.quantize(zdf)
    df = model.decode(zdf, pad_axes, condition=condition)["df"].squeeze(0)
    df = valdata_ae.denormalize(df=df, file_index=0)
    return df[[0, 1]] + df[[2, 3]]


results = []
with torch.no_grad():
    for traj in TRAJECTORIES:
        for t in TIMESTEPS[:-1]:
            ds_a = DATASETS[traj][t]["a"].to(device)
            ds_b = DATASETS[traj][t]["b"].to(device)
            ds_c = DATASETS[traj][t]["c"].to(device)

            gt = ds_c.full_df

            # extremes
            gt_a = ds_a.full_df
            gt_b = ds_b.full_df
            l1_a, psnr_a = compute_metrics(gt, gt_a)

            # Interpolated model (midpoint)
            model_a = MODELS[traj][t]
            model_b = MODELS[traj][t + DELTA]

            # weight space
            alpha = 0.5
            model_mid = interpolate_models(model_a, model_b, alpha)
            with torch.no_grad():
                pred_mid = sample_field(model_mid, ds_c, device=device, timestep=None)
            nf_l1_mid, nf_psnr_mid = compute_metrics(gt, pred_mid)

            # data space
            gt_avg = (1 - alpha) * gt_a + alpha * gt_b
            data_l1_mid, data_psnr_mid = compute_metrics(gt, gt_avg)

            # autoencoders
            traindata_ae = CycloneAEDataset(
                path=cfg.dataset.path,
                split="train",
                input_fields=["df", "phi", "flux"],
                random_seed=cfg.seed,
                normalization=cfg.dataset.normalization,
                normalization_scope=cfg.dataset.normalization_scope,
                trajectories=cfg.dataset.training_trajectories,
                separate_zf=cfg.dataset.separate_zf,
                real_potens=True,
                cond_filters=vars(cfg.dataset.training_cond_filters),
                stage="autoencoder",
                conditions=["itg", "dg", "s_hat", "q"],
            )
            valdata_ae = CycloneAEDataset(
                path="/restricteddata/ukaea/gyrokinetics/preprocessed",
                split="train",
                input_fields=["df", "phi", "flux"],
                random_seed=cfg.seed,
                normalization=cfg.dataset.normalization,
                normalization_scope=cfg.dataset.normalization_scope,
                normalization_stats=traindata_ae.norm_stats,
                trajectories=[f"{traj}.h5"],
                separate_zf=cfg.dataset.separate_zf,
                real_potens=True,
                stage="autoencoder",
                conditions=["itg", "dg", "s_hat", "q"],
            )

            sample_a = valdata_ae[t]
            sample_b = valdata_ae[t + DELTA]
            df_a = sample_a.df.unsqueeze(0).to(device)
            df_b = sample_b.df.unsqueeze(0).to(device)
            condition_a = sample_a.conditioning.unsqueeze(0).to(device)
            condition_b = sample_b.conditioning.unsqueeze(0).to(device)
            ae_zdf_a, ae_condition_a, ae_pad_axes = ae.encode(
                df_a, condition=condition_a
            )
            ae_zdf_b, ae_condition_b, _ = ae.encode(df_b, condition=condition_b)
            vqvae_zdf_a, vqvae_condition_a, vqvae_pad_axes = vqvae.encode(
                df_a, condition=condition_a
            )
            vqvae_zdf_b, vqvae_condition_b, _ = vqvae.encode(
                df_b, condition=condition_b
            )

            ae_df_ab = autoencoder_interp_latent(
                ae,
                lat_a=ae_zdf_a,
                lat_b=ae_zdf_b,
                condition_a=ae_condition_a,
                condition_b=ae_condition_b,
                pad_axes=ae_pad_axes,
                alpha=0.5,
            ).cpu()
            vqvae_df_ab = autoencoder_interp_latent(
                vqvae,
                lat_a=vqvae_zdf_a,
                lat_b=vqvae_zdf_b,
                condition_a=vqvae_condition_a,
                condition_b=vqvae_condition_b,
                pad_axes=vqvae_pad_axes,
                alpha=0.5,
            ).cpu()

            ae_l1_mid, ae_psnr_mid = compute_metrics(gt, ae_df_ab)
            vqvae_l1_mid, vqvae_psnr_mid = compute_metrics(gt, vqvae_df_ab)

            results.append(
                {
                    "Trajectory": traj,
                    "Timestep": t,
                    "Extremes L1": l1_a,
                    "Extremes PSNR": psnr_a,
                    "NF L1": nf_l1_mid,
                    "NF PSNR": nf_psnr_mid,
                    "AE L1": ae_l1_mid,
                    "AE PSNR": ae_psnr_mid,
                    "VQ-VAE l1": vqvae_l1_mid,
                    "VQ-VAE PSNR": vqvae_psnr_mid,
                    "Data L1": data_l1_mid,
                    "Data PSNR": data_psnr_mid,
                }
            )

df_metrics = pd.DataFrame(results)

In [ ]:
row = {
    "Extremes": (
        f"{df_metrics['Extremes PSNR'].mean():.1f}",
        f"{df_metrics['Extremes L1'].mean():.2f}",
    ),
    r"$\boldsymbol{f}$ (data)": (
        f"{df_metrics['Data PSNR'].mean():.1f}",
        f"{df_metrics['Data L1'].mean():.2f}",
    ),
    "NF (weights)": (
        f"{df_metrics['NF PSNR'].mean():.1f}",
        f"{df_metrics['NF L1'].mean():.2f}",
    ),
    "AE (latents)": (
        f"{df_metrics['AE PSNR'].mean():.1f}",
        f"{df_metrics['AE L1'].mean():.2f}",
    ),
    "VQ-VAE (latents)": (
        f"{df_metrics['VQ-VAE PSNR'].mean():.1f}",
        f"{df_metrics['VQ-VAE l1'].mean():.2f}",
    ),
}

# Build table
table = [
    r"\begin{tabular}{lcc}",
    r"\toprule",
    r"Model & PSNR & L1 \\",
    r"\midrule",
]

for model, (psnr, l1) in row.items():
    table.append(f"{model} & {psnr} & {l1} \\\\")

table += [
    r"\bottomrule",
    r"\end{tabular}",
]

table = "\n".join(table)

print(table)

## Physics interpretation tables

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("../grid_search_mlp_pinn.csv").sort_values(
    "pre_val/df psnr", ascending=False
)
df

In [ ]:
import pandas as pd

loss_map = {"recon": 0, "int": 1, "diag": 2, "grad": 4, "PINN": 5}

# Define the metrics columns for LaTeX table
metrics_cols = {
    "f": "df psnr",
    "Q": "flux loss",
    r"\phi": "phi loss",
    "k_y^{\\text{spec}}": "kyspec loss",
    "Q^{\\text{spec}}": "qspec loss",
}

# Initialize LaTeX table strings
latex_rows = []

for loss_name, idx in loss_map.items():
    row = df.loc[idx]

    # Choose pre_val for recon, fine_val for others
    prefix = "pre_val" if loss_name == "recon" else "fine_val"

    # Format PSNR values and other metrics
    metrics_values = []
    for col_name, metric in metrics_cols.items():
        val = row[f"{prefix}/{metric}"]
        if pd.isna(val):
            metrics_values.append("")
        else:
            # Round and strip trailing zeros
            val_str = f"{val:.2f}".rstrip("0").rstrip(".")
            metrics_values.append(val_str)

    # Decide if we color the cell
    if loss_name == "int":
        loss_cell = r"\cellcolor{green!10}{$+ \mathcal{L}_{\text{int}}$}"
    elif loss_name == "diag":
        loss_cell = r"\cellcolor{yellow!20}{$+ \mathcal{L}_{\text{diag}}$}"
    elif loss_name == "grad":
        loss_cell = r"\cellcolor{blue!10}{$+ \mathcal{L}_{\text{grad}}$}"
    elif loss_name == "recon":
        loss_cell = r"$\phantom{+}\mathcal{L}_{\text{recon}}$"
    else:
        loss_cell = r"$+ \mathcal{L}_{\text{PINN}}$"

    # Build the row string
    latex_row = f" & {loss_cell} & " + " & ".join(metrics_values) + r" \\"
    latex_rows.append(latex_row)

# Combine rows with multirow
latex_nf = "\n".join(latex_rows)
print(latex_nf)

In [ ]:
grid = pd.read_csv("../grid_search_siren.csv").sort_values(
    "pre_val/df psnr", ascending=False
)
grid

In [ ]:
caption = "SIREN grid search combinations."
label = "tab:siren_combinations"

df_fmt = grid.copy()
df_fmt["first_w0"] = df_fmt["first_w0"].map(
    lambda x: f"{x:.2f}".rstrip("0").rstrip(".")
)
df_fmt["hidden_w0"] = df_fmt["hidden_w0"].map(
    lambda x: f"{x:.2f}".rstrip("0").rstrip(".")
)
df_fmt[r"$w_0^{\text{initial}}$"] = df_fmt["first_w0"]
df_fmt[r"$w_0^{\text{hidden}}$"] = df_fmt["hidden_w0"]

df_fmt[r"PSNR$_\boldsymbol{f}$"] = (
    (df_fmt["pre_val/df psnr"] / 4)
    .round(2)
    .map(lambda x: f"{x:.2f}".rstrip("0").rstrip("."))
)

df_fmt["Embedding"] = df_fmt["embed_type"].replace(
    {"sincos_continuous": "SinCos", "discrete": "Discrete", "linear": "Linear"}
)
df_fmt["Skip"] = df_fmt["skips"].map({True: "Yes", False: "No"})
df_fmt["Learning rate"] = r"$5e{-3}$"
df_fmt["PSNR$_\boldsymbol{f}$"] = (
    (df_fmt["pre_val/df psnr"] / 4)
    .round(2)
    .map(lambda x: f"{x:.2f}".rstrip("0").rstrip("."))
)

table_df = df_fmt[
    [
        "Embedding",
        r"$w_0^{\text{initial}}$",
        r"$w_0^{\text{hidden}}$",
        "Skip",
        "Learning rate",
        r"PSNR$_\boldsymbol{f}$",
    ]
]

latex_code = table_df.to_latex(index=False, escape=False, column_format="lccccl")

latex_code = (
    "\\begin{table}[h]\n"
    "\\centering\n"
    "\\renewcommand{\\arraystretch}{1.2}\n"
    f"\\caption{{{caption}\\label{{{label}}}}}\n" + latex_code + "\\end{table}\n"
)
print(latex_code)

In [ ]:
grid = pd.read_csv("../grid_search_wire.csv").sort_values(
    "pre_val/df psnr", ascending=False
)
grid

In [ ]:
caption = "WIRE grid search combinations."
label = "tab:wire_combinations"

df_fmt = grid.copy()
df_fmt["first_w0"] = df_fmt["first_w0"].map(
    lambda x: f"{x:.2f}".rstrip("0").rstrip(".")
)
df_fmt["hidden_w0"] = df_fmt["hidden_w0"].map(
    lambda x: f"{x:.2f}".rstrip("0").rstrip(".")
)
df_fmt[r"$w_0^{\text{initial}}$"] = df_fmt["first_w0"]
df_fmt[r"$w_0^{\text{hidden}}$"] = df_fmt["hidden_w0"]
df_fmt["Embedding"] = df_fmt["embed_type"].replace(
    {"sincos_continuous": "SinCos", "discrete": "Discrete", "linear": "Linear"}
)
df_fmt["Learning rate"] = r"$1e{-2}$"
df_fmt[r"PSNR$_\boldsymbol{f}$"] = (
    (df_fmt["pre_val/df psnr"] / 4)
    .round(2)
    .map(lambda x: f"{x:.2f}".rstrip("0").rstrip("."))
)

table_df = df_fmt[
    [
        "Embedding",
        r"$w_0^{\text{initial}}$",
        r"$w_0^{\text{hidden}}$",
        "Learning rate",
        r"PSNR$_\boldsymbol{f}$",
    ]
]

latex_code = table_df.to_latex(index=False, escape=False, column_format="lccccl")

latex_code = (
    "\\begin{table}[h]\n"
    "\\centering\n"
    "\\renewcommand{\\arraystretch}{1.2}\n"
    f"\\caption{{{caption}\\label{{{label}}}}}\n" + latex_code + "\\end{table}\n"
)
print(latex_code)

In [ ]:
grid = pd.read_csv("../grid_search_mlp_.csv").sort_values(
    "pre_val/df psnr", ascending=False
)
grid

In [ ]:
caption = "MLP grid search combinations."
label = "tab:mlp_combinations"

df_fmt = grid.copy()
df_fmt["Activation"] = df_fmt["act_fn"].replace(
    {"silu": "SiLU", "relu": "ReLU", "gelu": "GELU"}
)
df_fmt["Embedding"] = df_fmt["embed_type"].replace(
    {"sincos_continuous": "SinCos", "discrete": "Discrete", "linear": "Linear"}
)
df_fmt["Skip"] = df_fmt["skips"].map({True: "Yes", False: "No"})
df_fmt["Learning rate"] = r"$5e{-3}$"
df_fmt[r"PSNR$_\boldsymbol{f}$"] = (
    (df_fmt["pre_val/df psnr"] / 4)
    .round(2)
    .map(lambda x: f"{x:.2f}".rstrip("0").rstrip("."))
)

table_df = df_fmt[
    ["Activation", "Embedding", "Skip", "Learning rate", r"PSNR$_\boldsymbol{f}$"]
]

latex_code = table_df.to_latex(
    index=False, escape=False, column_format="lccccl"  # allow math mode in headers
)

latex_code = (
    "\\begin{table}[h]\n"
    "\\centering\n"
    "\\renewcommand{\\arraystretch}{1.2}\n"
    f"\\caption{{{caption}\\label{{{label}}}}}\n" + latex_code + "\\end{table}\n"
)
print(latex_code)

## Timing experiments

In [ ]:
import os
import sys


sys.path.append("..")

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

device = "cuda"

import torch
import numpy as np
from einops import rearrange

import os
import pywt
import zfpy
from sklearn.decomposition import PCA
import glymur

In [ ]:
from torch import nn
import time

from neugk.pinc.neural_fields.nf_utils import sample_field
from neugk.pinc.neural_fields.nf_train import train_density, train_pinc
from neugk.pinc.neural_fields.models.mlp import MLPNF
from neugk.pinc.neural_fields.data import CycloneNFDataset, CycloneNFDataLoader


data = CycloneNFDataset(
    trajectory="iteration_13.h5",
    timesteps=200,
    normalize="zscore",
    normalize_coords=False,
    realpotens=True,
)
loader = CycloneNFDataLoader(data, 2048, preload=True, shuffle=True, pin_memory=True)

model = MLPNF(
    data.ndim,
    data.nchannels,
    n_layers=5,
    dim=64,
    act_fn=nn.SiLU,
    use_checkpoint=False,
    skips=True,
    embed_type="discrete",
)

optim = torch.optim.AdamW(model.parameters(), 5e-3, weight_decay=1e-7)
aux_optim = torch.optim.SGD(model.parameters(), 5e-4)

In [ ]:
runtimes = {}

In [ ]:
start = time.perf_counter_ns()
model, _, _ = train_density(
    model,
    optim=optim,
    n_epochs=5,
    data=data,
    loader=loader,
    device=device,
    use_print=False,
    use_tqdm=False,
)
model, _, _ = train_pinc(
    model,
    optim=optim,
    n_epochs=20,
    data=data,
    device=device,
    optim=aux_optim,
    pinc_loss_weight={
        "df": 1.0,
        "flux": 1.0,
        "phi": 0.1,
        "kyspec": 1.0,
        "qspec": 1.0,
        "kyspec monotonicity": 1.0,
        "qspec monotonicity": 1.0,
        "mass": 0.0,
    },
    use_print=False,
    use_tqdm=False,
)
compress_time = (time.perf_counter_ns() - start) / 1e6

start = time.perf_counter_ns()
_ = sample_field(model, data, device)
recon_time = (time.perf_counter_ns() - start) / 1e6

runtimes["NF"] = (compress_time, recon_time)
runtimes["NF"]

In [ ]:
from neugk.dataset import CycloneAEDataset
from neugk.pinc.autoencoders.ae_utils import load_autoencoder


ae_root = "/system/user/publicwork/gyrokinetics_checkpoints/compression_ckpts"

# AE
cr_1208 = f"{ae_root}/20250911_143423/best.pth"
ae, _, cfg = load_autoencoder(cr_1208, device=device)
cr_1208_peft = f"{ae_root}/20250920_190044/ckp.pth"
ae, _, ae_cfg = load_autoencoder(cr_1208_peft, device=device, load_peft=True)

# VQ-VAE
cr_77368 = f"{ae_root}/20250911_143815/best.pth"
vqvae, _, vqvae_cfg = load_autoencoder(cr_77368, device=device)
cr_77368_peft = f"{ae_root}/20250919_073133/best.pth"
vqvae, _, vqvae_cfg = load_autoencoder(cr_77368_peft, device=device, load_peft=True)

ae = ae.to(device)
vqvae = vqvae.to(device)
valdata_ae = CycloneAEDataset(
    path="/restricteddata/ukaea/gyrokinetics/preprocessed",
    split="train",
    input_fields=["df", "phi", "flux"],
    random_seed=cfg.seed,
    normalization=None,
    trajectories=["iteration_13.h5"],
    separate_zf=cfg.dataset.separate_zf,
    real_potens=True,
    stage=cfg.stage,
    conditions=["itg", "dg", "s_hat", "q"],
)

start = time.perf_counter_ns()
# ae reconstruct
sample = valdata_ae[200]
df = sample.df.unsqueeze(0).to(device)
condition = sample.conditioning.unsqueeze(0).to(device)
z, cond, pad = ae.encode(df, condition=condition)

compress_time = (time.perf_counter_ns() - start) / 1e6

start = time.perf_counter_ns()
ae_df = ae.decode(z, pad, condition=cond)["df"]
if ae_df.shape[0] == 4:
    ae_df = ae_df[[0, 1]] + ae_df[[2, 3]]

recon_time = (time.perf_counter_ns() - start) / 1e6

runtimes["AE"] = (compress_time, recon_time)

start = time.perf_counter_ns()
# ae reconstruct
sample = valdata_ae[200]
df = sample.df.unsqueeze(0).to(device)
condition = sample.conditioning.unsqueeze(0).to(device)
z, cond, pad = vqvae.encode(df, condition=condition)

compress_time = (time.perf_counter_ns() - start) / 1e6

start = time.perf_counter_ns()
ae_df = vqvae.decode(z, pad, condition=cond)["df"]
if ae_df.shape[0] == 4:
    ae_df = ae_df[[0, 1]] + ae_df[[2, 3]]

recon_time = (time.perf_counter_ns() - start) / 1e6

runtimes["VQ-VAE"] = (compress_time, recon_time)

In [ ]:
df = data.full_df

start = time.perf_counter_ns()

vp, s = df.shape[1], df.shape[3]
df = rearrange(df, "c vp vm s x y -> c (vp vm) (s y) x").cpu().numpy()
zfp_compressed = zfpy.compress_numpy(df, tolerance=2000)

compress_time = (time.perf_counter_ns() - start) / 1e6

start = time.perf_counter_ns()

zf_df = zfpy.decompress_numpy(zfp_compressed)
zf_df = rearrange(zf_df, "c (vp vm) (s y) x -> c vp vm s x y", vp=vp, s=s)
zf_df = torch.from_numpy(zf_df)

recon_time = (time.perf_counter_ns() - start) / 1e6

runtimes["ZFP"] = (compress_time, recon_time)

df = data.full_df


start = time.perf_counter_ns()

df = df.cpu().numpy()
vp, vm, s, x, y = df.shape[1:]

coeffs = []
for c in range(2):
    dec = pywt.wavedecn(df[c], wavelet="haar", mode="periodization", level=1)
    coeff, slices = pywt.coeffs_to_array(dec)
    coeff[np.abs(coeff) < 22] = 0
    coeffs.append((coeff, slices))

compress_time = (time.perf_counter_ns() - start) / 1e6

start = time.perf_counter_ns()

wt_df = []
for coeff, slices in coeffs:
    recon = pywt.array_to_coeffs(coeff, slices, output_format="wavedecn")
    recon = pywt.waverecn(recon, wavelet="haar", mode="periodization")
    wt_df.append(recon)

wt_df = np.stack(wt_df, axis=0)
wt_df = wt_df[:, :vp, :vm, :s, :x, :y]
wt_df = torch.from_numpy(wt_df)

recon_time = (time.perf_counter_ns() - start) / 1e6

runtimes["Wavelet"] = (compress_time, recon_time)


df = data.full_df

vp, vm, s, x, y = df.shape[1:]
df_np = rearrange(df, "c vp vm s x y -> c (vp vm s) (x y)").cpu().numpy()

pca_results = []
compressed = []
compressed_size = 0

# ------------------------
# compression timer start
# ------------------------
start = time.perf_counter_ns()

pcas = []  # keep fitted PCA objects for later reconstruction
for c in range(2):  # assume 2 components
    pca = PCA(n_components=2)
    transformed = pca.fit_transform(df_np[c])
    pcas.append(pca)

    # store compressed form (components + params)
    compressed_version = {
        "components": transformed,
        "mean": pca.mean_,
        "explained_variance": pca.explained_variance_,
    }
    compressed.append(compressed_version)

    compressed_size += (
        transformed.nbytes + pca.mean_.nbytes + pca.explained_variance_.nbytes
    )

compress_time = (time.perf_counter_ns() - start) / 1e6

# ------------------------
# reconstruction timer start
# ------------------------
start = time.perf_counter_ns()

for c in range(2):
    reconstructed = pcas[c].inverse_transform(compressed[c]["components"])
    pca_results.append(reconstructed)

pca_df = np.stack(pca_results, axis=0)

# reshape back
pca_df = rearrange(
    pca_df,
    "c (vp vm s) (x y) -> c vp vm s x y",
    vp=vp,
    vm=vm,
    s=s,
    x=x,
    y=y,
)
pca_df = torch.from_numpy(pca_df)

recon_time = (time.perf_counter_ns() - start) / 1e6

runtimes["PCA"] = (compress_time, recon_time)


start = time.perf_counter_ns()

c, vp, vm, _, x, _ = df.shape
df_np = df.cpu().numpy()

df_flat = rearrange(df_np, "c vp vm s x y -> c (vp vm s) (x y)")

compressed_data = []
compressed_size = 0.0
recon_flat = np.zeros_like(df_flat)

for ch in range(c):
    slice_ = df_flat[ch]

    mn, mx = slice_.min(), slice_.max()
    if mx - mn == 0:
        norm_slice = np.zeros_like(slice_)
    else:
        norm_slice = (slice_ - mn) / (mx - mn)

    img_uint16 = (norm_slice * 65535).astype(np.uint16)

    glymur.Jp2k("/tmp/df.jp2", data=img_uint16, cratios=[100.0 / 0.1])
    compressed_data.append({"bytes": None, "min": mn, "max": mx})
    compressed_size += os.path.getsize("/tmp/df.jp2")

compress_time = (time.perf_counter_ns() - start) / 1e6

start = time.perf_counter_ns()

for ch in range(c):
    jp2 = glymur.Jp2k("/tmp/df.jp2")
    recon_uint16 = jp2[:]
    mn, mx = compressed_data[ch]["min"], compressed_data[ch]["max"]
    recon_norm = recon_uint16.astype(np.float32) / 65535.0
    recon_flat[ch] = recon_norm * (mx - mn) + mn

recon_np = rearrange(
    recon_flat, "c (vp vm s) (x y) -> c vp vm s x y", vp=vp, vm=vm, x=x
)
recon_np = torch.from_numpy(recon_np)

recon_time = (time.perf_counter_ns() - start) / 1e6

runtimes["JPEG2000"] = (compress_time, recon_time)

In [ ]:
runtimes